In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import warnings
from scipy.stats import norm, chi2
import numpy as np
import plotly.graph_objects as go
from ipywidgets import interact, Dropdown
warnings.filterwarnings('ignore')

In [2]:
def downloaded_data_validation(prices):

    validity = []
    warning = False

    duplicate_dates = prices.index.duplicated().sum()

    for asset in prices.columns:

        total_rows = len(prices[asset])
        missing = prices[asset].isna().sum()
        missing_pct = (missing / total_rows) * 100 if total_rows else 0

        # Asset-specific invalid price check
        invalid_prices = (
            (prices[asset] <= 0) |
            (~np.isfinite(prices[asset]))
        ).sum()

        validity.append({
            'Ticker': asset,
            'Observations': total_rows,
            'Missing': missing,
            'Missing %': missing_pct,
            'Invalid Prices': invalid_prices,
            'Duplicate Dates': duplicate_dates,
            'First Date': prices.index.min(),
            'Last Date': prices.index.max()
        })

        if missing > 0 or invalid_prices > 0 or duplicate_dates > 0:
            warning = True

    validity = pd.DataFrame(validity)

    return validity, warning

In [3]:

def download_data(tickers, period='10y', interval='1d'):
    data = yf.download(tickers=tickers, period=period, interval=interval, auto_adjust=True)['Close']

    if isinstance(data, pd.Series): #incase the tickers is a single asset
        data = data.to_frame()
        
    missing = data.isna().sum()

    if missing.any():
        print(f'Missing Observations : {missing} data points\n Interpolating Linerarly...')
        data.interpolate(inplace=True)
    
    validity, warning = downloaded_data_validation(data)

    return data, validity, warning



In [4]:
def calcualte_portfolio_returns(prices, weights, rebalance_frequency = "Daily"):
    pct_change = prices.pct_change().dropna()

    allocation_drifts = pd.DataFrame(index=prices.index, columns= prices.columns)

    
    if rebalance_frequency == 'Daily':
        portfolio_daily_returns = pct_change @ weights 
        allocation_drifts.iloc[:,:] = weights.values

    elif rebalance_frequency =='BuynHold':
        portfolio_value = (prices/prices.iloc[0]) @ weights
        portfolio_daily_returns = portfolio_value.pct_change().dropna()

        asset_values = (prices / prices.iloc[0]).mul(weights, axis=1)

        allocation_drifts = (asset_values.shift(1).div(asset_values.shift(1).sum(axis=1), axis=0).loc[pct_change.index])

    elif rebalance_frequency == "Monthly":

        portfolio_daily_returns = pd.Series(index=pct_change.index, dtype=float)

        current_weights = weights.copy()
        previous_month = None

        for date, daily_returns in pct_change.iterrows():

            current_month = date.to_period("M")

            if previous_month is not None and current_month != previous_month:
                current_weights = weights.copy()

            allocation_drifts.loc[date] = current_weights

            portfolio_daily_returns.loc[date] = daily_returns @ current_weights

            current_weights *= (1 + daily_returns)
            current_weights /= current_weights.sum()

            previous_month = current_month

    else:
        raise ValueError("Invalid rebalancing frequency.")

    return portfolio_daily_returns, allocation_drifts




In [5]:
def calcualte_portfolio_value(initial_investment, portfolio_daily_returns):
    portfolio_value = (1 + portfolio_daily_returns).cumprod() * initial_investment
    return portfolio_value

In [6]:
def calculate_returns_metrics(portfolio_value):
    initial_value = portfolio_value.iloc[0]
    final_value = portfolio_value.iloc[-1]
    years = (len(portfolio_value) - 1) / 252

    total_return = (final_value / initial_value) - 1
    cagr = np.pow(final_value / initial_value, 1/years) - 1

    return total_return, cagr

def calculate_volatility_metrics(portfolio_returns):
    daily_vol = portfolio_returns.std()
    annual_vol = daily_vol * (252**0.5)

    return daily_vol, annual_vol


def calculate_portfolio_metrics(portfolio_value, risk_free_rate = 0.02):    

    portfolio_returns = portfolio_value.pct_change().dropna()

    total_return, cagr = calculate_returns_metrics(portfolio_value)
    daily_vol, annual_vol = calculate_volatility_metrics(portfolio_returns)

    sharpe = ((portfolio_returns.mean() * 252)  - risk_free_rate) / annual_vol

    metrics = {
        'Total Return': total_return,
        'CAGR': cagr,
        'Daily Volatility': daily_vol,
        'Annualized Volatility': annual_vol,
        'Sharpe Ratio': sharpe
    }

    return metrics

In [7]:
tickers = ['AAPL', 'GOOG','WMT', 'GLD']
period = '10y'
interval = '1d'
prices, validity, warning = download_data(tickers)
investment_start = '2021-01-01'
investment_end = '2025-01-01'

investment_prices = prices[(prices.index>=investment_start) & (prices.index<=investment_end)]
weights = pd.Series([0.2,0.1,0.3,0.4], index = prices.columns)
rebalance_frequency = 'BuynHold'
portfolio_daily_returns, allocation_drifts = calcualte_portfolio_returns(investment_prices, weights=weights, rebalance_frequency=rebalance_frequency)
initial_investment = 100_000
portfolio_value = calcualte_portfolio_value(initial_investment=initial_investment, portfolio_daily_returns=portfolio_daily_returns)
risk_free_rate = 0.04
metrics = calculate_portfolio_metrics(portfolio_value, risk_free_rate)



[*********************100%***********************]  4 of 4 completed


In [8]:
metrics

{'Total Return': np.float64(0.9709508235380846),
 'CAGR': np.float64(0.1858675634070217),
 'Daily Volatility': np.float64(0.011407667190311058),
 'Annualized Volatility': np.float64(0.181091102549724),
 'Sharpe Ratio': np.float64(0.8113358604846972)}

In [9]:
import plotly.express as px
comparison = pd.DataFrame()

normalized_prices = (investment_prices/investment_prices.iloc[0]) * initial_investment

comparison = normalized_prices.copy()
comparison['Portfolio'] = portfolio_value
comparison.iloc[0, comparison.columns.get_loc("Portfolio")] = initial_investment

In [10]:
fig = px.line(
    comparison,
    x = comparison.index,
    y = comparison.columns,
    title="Portfolio Performance vs Individual Assets",
    labels={
        "x": 'date',
        'value': 'Portfolio Values($)',
        'variable' : 'Asset'
    }
)
for trace in fig.data:
    if trace.name != "Portfolio":
        trace.opacity = 0.3
    else:
        trace.opacity = 1
        trace.line.width = 3

fig.show()

In [11]:
fig = px.area(
    allocation_drifts * 100,
    x = allocation_drifts.index,
    y = allocation_drifts.columns,
    labels = {
        'x': 'Date',
        'value': 'Allocation (%)',
        'variable': 'Asset'
    }
)
fig.update_layout(
    yaxis_title="Allocation (%)",
    yaxis_range=[0, 100],
    hovermode="x unified",
    legend_title="Asset"
)
fig.show()

### Portfolio Construction complete
# Market Data And Analytics

In [12]:
def calculate_asset_statistics(prices):

    asset_data = []

    for ticker in prices.columns:

        asset_prices = prices[ticker]
        missing = asset_prices.isna().sum()
        observations = asset_prices.notna().sum()
        missing_pct = missing / len(prices) * 100 if len(prices) else 0

        invalid_prices = ((asset_prices <= 0) | (~np.isfinite(asset_prices) & asset_prices.notna())).sum()

        valid_prices = asset_prices.dropna()

        asset_data.append({
            'ticker': ticker,
            'first_date': asset_prices.first_valid_index(),
            'last_date': asset_prices.last_valid_index(),
            'missing_observations': missing,
            'missing_percentage': missing_pct,
            'number_of_observations': observations,
            'data_completeness_percentage': 100 - missing_pct,
            'latest_valid_price': valid_prices.iloc[-1] if not valid_prices.empty else np.nan,
            'latest_valid_price_date': asset_prices.last_valid_index(),
            'invalid_prices': invalid_prices,
            'data_validation_warning': missing > 0 or invalid_prices > 0
        })

    return pd.DataFrame(asset_data).set_index('ticker')

In [13]:
def calculate_asset_returns(prices):
    daily_returns = prices.pct_change(fill_method=None)
    return daily_returns


def calculate_return_statistics(daily_returns):

    asset_data = []

    for ticker in daily_returns.columns:

        asset_returns = daily_returns[ticker].dropna()

        daily_mean = asset_returns.mean()
        annualized_return = daily_mean * 252

        daily_vol = asset_returns.std()
        annualized_vol = daily_vol * np.sqrt(252)

        asset_data.append({
            'ticker': ticker,
            'daily_mean_return': daily_mean,
            'annualized_return': annualized_return,
            'daily_volatility': daily_vol,
            'annualized_volatility': annualized_vol,
            'skewness': asset_returns.skew(),
            'excess_kurtosis': asset_returns.kurt(),
            'best_daily_return': asset_returns.max(),
            'worst_daily_return': asset_returns.min()
        })

    return pd.DataFrame(asset_data).set_index('ticker')

In [14]:
from arch import arch_model

def calculate_rolling_vol(asset_return, window, annualization_factor=252):
    return asset_return.rolling(window).std() * np.sqrt(annualization_factor)

def calculate_ewma_vol(asset_return, lambda_, window, annualization_factor=252):
    ewma_vol = np.sqrt(asset_return.pow(2).ewm(alpha = 1-lambda_, min_periods = window, adjust = False).mean())
    return ewma_vol * np.sqrt(annualization_factor)

def calculate_garch_vol(asset_return, annualization_factor=252):
    returns = asset_return.dropna() * 100
    model = arch_model(returns, mean = 'Zero', vol = 'GARCH', p = 1, q = 1, dist = 'normal')

    fitted_model = model.fit(disp = 'off')

    garch_vol = fitted_model.conditional_volatility / 100
    return garch_vol * np.sqrt(annualization_factor)



def vol_analysis(daily_returns, window = 30, lambda_ = 0.94):
    vol_stats = {}

    for ticker in daily_returns.columns:
        rolling_vol = calculate_rolling_vol(daily_returns[ticker], window)
        ewma_vol = calculate_ewma_vol(daily_returns[ticker], lambda_, window)
        garch_vol = calculate_garch_vol(daily_returns[ticker])
        ticker_vol = pd.concat(
            [
                rolling_vol.rename('rolling_vol'),
                ewma_vol.rename('ewma_vol'),
                garch_vol.rename('garch_vol')
            ], axis=1
        )
        vol_stats[ticker] = ticker_vol

    annual_asset_vol = daily_returns.std() * np.sqrt(252)

    return vol_stats, annual_asset_vol




In [15]:
def calculate_correlation_matrix(daily_returns, window = None):
    if window is None:
        return daily_returns.corr()

    daily_returns = daily_returns.iloc[-window:,:]
    return daily_returns.corr()


def calculate_rolling_correlation(daily_returns, asset1, asset2, window = 60):
    return daily_returns[asset1].rolling(window).corr(daily_returns[asset2])





In [16]:
from scipy.stats import jarque_bera

def calculate_normality_test(asset_returns, significance_level = 0.05):
    normality = []

    for ticker in asset_returns.columns:

        jb_statistic, p_value = jarque_bera(asset_returns[ticker].dropna())

        normality.append({
            'ticker':ticker,
            'jb_statistic':jb_statistic,
            'p_value': p_value,
            'is_normal': p_value >=significance_level
        })
    normality = pd.DataFrame(normality).set_index('ticker')
    return normality



In [17]:
daily_returns = calculate_asset_returns(investment_prices)

return_statistics = calculate_return_statistics(daily_returns)

asset_statistics = calculate_asset_statistics(investment_prices)

vol_stats, annual_asset_vol = vol_analysis(daily_returns, window=30, lambda_=0.94)

correlation_matrix = calculate_correlation_matrix(daily_returns)

normality_results = calculate_normality_test(daily_returns)

In [18]:
vol_colors = {
    "rolling_vol": "#636EFA",
    "ewma_vol": "#EF553B",
    "garch_vol": "#00CC96"
}

vol_labels = {
    "rolling_vol": "Rolling Volatility",
    "ewma_vol": "EWMA Volatility",
    "garch_vol": "GARCH Volatility"
}

def plot_asset_volatility(ticker, vol_stats, annual_asset_vol):

    asset_vol = vol_stats[ticker]

    fig = go.Figure()

    for column in ["rolling_vol", "ewma_vol", "garch_vol"]:

        fig.add_trace(
            go.Scatter(
                x=asset_vol.index,
                y=asset_vol[column],
                name=vol_labels[column],
                mode="lines",
                line=dict(
                    color=vol_colors[column],
                    width=1.5
                )
            )
        )

    # Historical volatility reference line
    fig.add_hline(
        y=annual_asset_vol[ticker],
        line_dash="dash",
        line_color="black",
        line_width=1.5,
        annotation_text="Historical Volatility",
        annotation_position="top left"
    )

    fig.update_layout(
        title=f"{ticker} - Volatility Analysis",
        template="plotly_white",
        height=400,
        hovermode="x unified",
        yaxis_title="Annualized Volatility",
        yaxis_tickformat=".0%",
        legend=dict(
            orientation="h",
            y=1.12,
            x=0.5,
            xanchor="center"
        ),
        margin=dict(l=50, r=20, t=90, b=40)
    )

    return fig


asset_figures = {}

for ticker in daily_returns.columns:

    asset_figures[ticker] = plot_asset_volatility(
        ticker,
        vol_stats,
        annual_asset_vol
    )
    asset_figures[ticker].show()


In [19]:


def plot_return_distribution(ticker, daily_returns):

    asset_returns = daily_returns[ticker].dropna()

    mean = asset_returns.mean()
    std = asset_returns.std()

    x = np.linspace(
        asset_returns.min(),
        asset_returns.max(),
        300
    )

    normal_pdf = norm.pdf(x, mean, std)

    fig = go.Figure()

    # Return histogram
    fig.add_trace(
        go.Histogram(
            x=asset_returns,
            histnorm="probability density",
            name="Actual Returns",
            opacity=0.65
        )
    )

    # Normal distribution curve
    fig.add_trace(
        go.Scatter(
            x=x,
            y=normal_pdf,
            mode="lines",
            name="Normal Distribution",
            line=dict(color="red", width=2)
        )
    )

    # Mean return
    fig.add_vline(
        x=mean,
        line_dash="dash",
        line_color="black",
        annotation_text="Mean"
    )

    fig.update_layout(
        title=f"{ticker} - Daily Return Distribution",
        template="plotly_white",
        height=400,
        xaxis_title="Daily Return",
        yaxis_title="Density",
        xaxis_tickformat=".1%"
    )

    return fig


asset_return_figures = {}

for ticker in daily_returns.columns:

    asset_return_figures[ticker] = plot_return_distribution(
        ticker,
        daily_returns
    )

    asset_return_figures[ticker].show()




In [20]:
def plot_asset_returns(ticker, daily_returns):

    asset_returns = daily_returns[ticker].dropna()

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=asset_returns.index,
            y=asset_returns,
            mode="lines",
            name="Daily Returns",
            line=dict(
                color="#636EFA",
                width=1
            )
        )
    )

    # Zero-return reference line
    fig.add_hline(
        y=0,
        line_dash="dash",
        line_color="black",
        line_width=1
    )

    fig.update_layout(
        title=f"{ticker} - Daily Returns",
        template="plotly_white",
        height=400,
        hovermode="x unified",
        xaxis_title="Date",
        yaxis_title="Daily Return",
        yaxis_tickformat=".1%"
    )

    return fig


asset_return_timeseries = {}

for ticker in daily_returns.columns:

    asset_return_timeseries[ticker] = plot_asset_returns(
        ticker,
        daily_returns
    )

    asset_return_timeseries[ticker].show()


In [21]:
def plot_correlation_heatmap(correlation_matrix):

    fig = px.imshow(
        correlation_matrix,
        text_auto=".2f",
        color_continuous_scale="RdBu_r",
        zmin=-1,
        zmax=1,
        aspect="auto",
        title="Asset Correlation Matrix"
    )

    fig.update_layout(
        template="plotly_white",
        height=600,
        width=800,
        coloraxis_colorbar=dict(
            title="Correlation"
        ),
        xaxis_title="Asset",
        yaxis_title="Asset"
    )

    return fig
correlation_fig = plot_correlation_heatmap(correlation_matrix)

correlation_fig.show()


In [22]:

@interact(
    lookback=Dropdown(
        options={
            "Full History": None,
            "250 Days": 250,
            "500 Days": 500,
            "750 Days": 750,
            "1000 Days": 1000
        },
        description="Lookback:"
    )
)
def update_correlation(lookback):

    corr = calculate_correlation_matrix(
        daily_returns,
        window=lookback
    )

    fig = plot_correlation_heatmap(corr)

    fig.show()

interactive(children=(Dropdown(description='Lookback:', options={'Full History': None, '250 Days': 250, '500 D…

In [23]:
def plot_rolling_correlation(rolling_corr, asset1, asset2, window):

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=rolling_corr.index,
        y=rolling_corr,
        mode="lines",
        name="Rolling Correlation",
        line=dict(width=1.5)
    ))

    fig.add_hline(
        y=0,
        line_dash="dash",
        line_color="black"
    )

    fig.update_layout(
        title=f"{window}-Day Rolling Correlation: {asset1} vs {asset2}",
        template="plotly_white",
        height=450,
        hovermode="x unified",
        xaxis_title="Date",
        yaxis_title="Correlation",
        yaxis=dict(range=[-1, 1])
    )

    return fig

@interact(
    asset1=Dropdown(
        options=prices.columns.tolist(),
        description="Asset 1:"
    ),
    asset2=Dropdown(
        options=prices.columns.tolist(),
        description="Asset 2:"
    ),
    window=Dropdown(
        options=[30, 60, 90, 120, 252],
        value=60,
        description="Window:"
    )
)
def display_rolling_correlation(asset1, asset2, window):

    if asset1 == asset2:
        print("Please select two different assets.")
        return

    rolling_corr = calculate_rolling_correlation(
        daily_returns, asset1, asset2, window
    )

    fig = plot_rolling_correlation(
        rolling_corr, asset1, asset2, window
    )

    fig.show()

interactive(children=(Dropdown(description='Asset 1:', options=('AAPL', 'GLD', 'GOOG', 'WMT'), value='AAPL'), …

In [24]:
def plot_return_boxplot(daily_returns):

    fig = go.Figure()

    for ticker in daily_returns.columns:
        fig.add_trace(go.Box(
            y=daily_returns[ticker].dropna(),
            name=ticker,
            boxpoints="outliers"
        ))

    fig.update_layout(
        title="Daily Return Distribution by Asset",
        template="plotly_white",
        height=500,
        yaxis_title="Daily Return",
        yaxis_tickformat=".1%",
        showlegend=False
    )

    return fig

return_boxplot = plot_return_boxplot(daily_returns)
return_boxplot.show()

In [25]:
def _calculate_holding_period_returns(portfolio_returns: pd.Series, holding_period: int) -> pd.Series:
    """Calculate overlapping compounded holding-period returns."""

    if holding_period < 1:
        raise ValueError("holding_period must be at least 1.")

    if portfolio_returns.isna().any():
        raise ValueError("portfolio_returns contains missing values.")

    if (portfolio_returns <= -1).any():
        raise ValueError("Simple returns cannot be less than or equal to -100%.")

    holding_period_returns = ((1 + portfolio_returns).rolling(window=holding_period).apply(np.prod, raw=True) - 1)

    return holding_period_returns.dropna()

def historical_var_es(portfolio_daily_returns, investment_value, confidence_level = 0.95, window = None, holding_period = 1):
    
    if window is not None:
        portfolio_daily_returns = portfolio_daily_returns.iloc[-window:]

    holding_period_returns = _calculate_holding_period_returns(portfolio_returns=portfolio_daily_returns,holding_period=holding_period)

    alpha = 1 - confidence_level
    var_return = -portfolio_daily_returns.quantile(alpha)
    var_amount = var_return * investment_value

    es_return = portfolio_daily_returns[portfolio_daily_returns<=-var_return].mean()
    es_amount = es_return * investment_value

    return {
        "var_return": var_return,
        "var_amount": var_amount,
        "es_return": es_return,
        "es_amount": es_amount,
        "observations": len(holding_period_returns),
        "confidence_level": confidence_level,
        "holding_period": holding_period
    }

def parametric_var_es(portfolio_daily_returns, investment_value, confidence_level = 0.95, window = None, holding_period = 1):

    if window is not None:
        portfolio_daily_returns = portfolio_daily_returns.iloc[-window:]

    alpha = 1 - confidence_level
    mu = portfolio_daily_returns.mean()
    sigma = portfolio_daily_returns.std()
    z_score = norm.ppf(alpha)
    var_return = -(mu * holding_period + sigma * z_score * np.sqrt(holding_period))
    var_amount = var_return * investment_value

    es_return = -holding_period * mu + np.sqrt(holding_period) * sigma * norm.pdf(z_score) / alpha
    es_amount = es_return * investment_value

    return {
        "var_return": var_return,
        "var_amount": var_amount,
        "es_return": es_return,
        "es_amount": es_amount,
        "observations": len(portfolio_daily_returns),
        "confidence_level": confidence_level,
        "holding_period": holding_period
    }

def monte_carlo(daily_returns, weights, investment_value, confidence_level = 0.95, n_simulations = 10000, window = None, holding_period = 1, random_seed = 42):

    if window is not None:
        daily_returns = daily_returns.iloc[-window:]

    alpha = 1 - confidence_level
    mean_returns = daily_returns.mean().values
    covariance_matrix = daily_returns.cov().values
    weights = np.asarray(weights, dtype=float)

    rng = np.random.default_rng(random_seed)
    simulated_asset_returns = rng.multivariate_normal(
        mean = mean_returns,
        cov = covariance_matrix,
        size = (n_simulations, holding_period)
    )

    simulated_portfolio_returns = simulated_asset_returns @ weights

    holding_period_returns = np.prod(1 + simulated_portfolio_returns, axis=1) - 1

    return_threshold = np.quantile(holding_period_returns, alpha)
    var_return = -return_threshold
    var_amount = var_return * investment_value

    tail_returns = holding_period_returns[holding_period_returns <= return_threshold]
    
    es_return = -tail_returns.mean()
    es_amount = es_return * investment_value

    return {
    "var_return": var_return,
    "var_amount": var_amount,
    "es_return": es_return,
    "es_amount": es_amount,
    "observations": len(holding_period_returns),
    "confidence_level": confidence_level,
    "holding_period": holding_period
}




In [26]:
from arch import arch_model

def return_historical_var_series(portfolio_returns, window_size = 250, confidence_level = 0.95):

    alpha = 1 - confidence_level    
    return_threshold = portfolio_returns.rolling(window_size).quantile(alpha).shift(1)
    var_series = -return_threshold

    return var_series

def return_parametric_var_series(portfolio_returns, window_size = 250, confidence_level = 0.95):

    alpha = 1 - confidence_level
    return_threshold = (portfolio_returns.rolling(window_size).mean() + (norm.ppf(alpha) * portfolio_returns.rolling(window_size).std())).shift(1)
    var_series = -return_threshold

    return var_series

def return_ewma_var_series(portfolio_returns, lambda_ = 0.94, window_size = 250, confidence_level = 0.95):

    alpha = 1 - confidence_level
    ewma_vol = portfolio_returns.pow(2).ewm(alpha = 1-lambda_, min_periods = window_size, adjust = False).mean().pow(0.5)
    return_threshold = (portfolio_returns.rolling(window_size).mean() + norm.ppf(alpha) * ewma_vol).shift(1)
    var_series = -return_threshold

    return var_series

def return_garch_var_series(portfolio_returns, window_size=250, confidence_level=0.95):

    alpha = 1 - confidence_level
    var_series = pd.Series(np.nan, index=portfolio_returns.index)

    training_returns = portfolio_returns.iloc[:window_size] * 100

    model = arch_model(training_returns, mean="Constant", vol="GARCH", p=1, q=1, dist="normal", rescale=False)
    fitted_model = model.fit(disp="off")

    params = fitted_model.params

    mu = params["mu"]
    omega = params["omega"]
    alpha_garch = params["alpha[1]"]
    beta_garch = params["beta[1]"]

    conditional_variance = fitted_model.conditional_volatility.iloc[-1] ** 2

    for i in range(window_size, len(portfolio_returns)):

        previous_return = portfolio_returns.iloc[i - 1] * 100
        residual = previous_return - mu

        conditional_variance = omega + alpha_garch * residual ** 2 + beta_garch * conditional_variance

        forecast_vol = np.sqrt(conditional_variance)

        return_threshold = mu + norm.ppf(alpha) * forecast_vol
        var_series.iloc[i] = -return_threshold / 100

    return var_series

In [27]:
import time

start = time.perf_counter()

garch_var = return_garch_var_series(
    portfolio_daily_returns,
    window_size=250,
    confidence_level=0.95
)

print(f"Execution time: {time.perf_counter() - start:.2f} seconds")

Execution time: 0.32 seconds


In [28]:
def calculate_var_exceptions(portfolio_returns, var_series):
    aligned_data = pd.concat(
        [portfolio_returns.rename('returns'), var_series.rename('var')]
        , axis = 1
    ).dropna()

    exceptions = aligned_data['returns']<-aligned_data['var']
    exceptions_count = exceptions.sum()
    exceptions_rate = exceptions.mean()

    return {
        'exceptions': exceptions,
        'exceptions_count': exceptions_count,
        'exceptions_rate': exceptions_rate
    }



In [29]:
def kupiec_test(exception_count, observations, confidence_level = 0.95):
    alpha = 1 - confidence_level
    exception_rate = exception_count / observations
    likelihood_null = np.pow((1-alpha), observations - exception_count) * np.pow(alpha,exception_count)
    likelihood_alternatve = np.pow((1-exception_rate), observations - exception_count) * np.pow(exception_rate,exception_count)

    lr = -2* np.log(likelihood_null/likelihood_alternatve)
    p_value = chi2.sf(lr, df=1)

    return {
        "lr_statistic": lr,
        "p_value": p_value,
        "reject_null": p_value < 0.05,
        "observed_rate": exception_rate,
        "expected_rate": alpha
    }


In [30]:
def christoffersen_test(exceptions):

    previous = exceptions.iloc[:-1].astype(int).values
    current = exceptions.iloc[1:].astype(int).values

    n00 = np.sum((previous == 0) & (current == 0))
    n01 = np.sum((previous == 0) & (current == 1))
    n10 = np.sum((previous == 1) & (current == 0))
    n11 = np.sum((previous == 1) & (current == 1))

    pi_0 = n01 / (n00 + n01) if (n00 + n01) > 0 else 0
    pi_1 = n11 / (n10 + n11) if (n10 + n11) > 0 else 0

    pi = (n01 + n11) / (n00 + n01 + n10 + n11)

    likelihood_independent = (
        (1 - pi) ** (n00 + n10) * pi ** (n01 + n11)
    )

    likelihood_markov = (
        (1 - pi_0) ** n00 * pi_0 ** n01
        * (1 - pi_1) ** n10 * pi_1 ** n11
    )

    lr = -2 * np.log(likelihood_independent / likelihood_markov)
    p_value = chi2.sf(lr, df=1)

    return {
        "n00": n00,
        "n01": n01,
        "n10": n10,
        "n11": n11,
        "pi_0": pi_0,
        "pi_1": pi_1,
        "lr_statistic": lr,
        "p_value": p_value,
        "reject_null": p_value < 0.05
    }

In [31]:
def conditional_coverage_test(kupiec_result, christoffersen_result):

    lr = kupiec_result["lr_statistic"] + christoffersen_result["lr_statistic"]
    p_value = chi2.sf(lr, df=2)

    return {
        "lr_statistic": lr,
        "p_value": p_value,
        "reject_null": p_value < 0.05
    }

In [32]:
def backtest_var(portfolio_returns,method="historical",window_size=60,confidence_level=0.95,lambda_=0.94):

    methods = {
        "historical": return_historical_var_series,
        "parametric": return_parametric_var_series,
        "ewma": return_ewma_var_series,
        "garch": return_garch_var_series
    }

    if method not in methods:
        raise ValueError(f"Unsupported VaR method: {method}")

    if method == "ewma":
        var_series = methods[method](
            portfolio_returns,
            lambda_=lambda_,
            window_size=window_size,
            confidence_level=confidence_level
        )
    else:
        var_series = methods[method](
            portfolio_returns,
            window_size=window_size,
            confidence_level=confidence_level
        )

    exception_results = calculate_var_exceptions(
        portfolio_returns,
        var_series
    )

    exceptions = exception_results["exceptions"]
    exception_count = exception_results["exceptions_count"]
    exception_rate = exception_results["exceptions_rate"]
    observations = len(exceptions)

    kupiec_results = kupiec_test(
        exception_count,
        observations,
        confidence_level
    )

    christoffersen_results = christoffersen_test(exceptions)

    conditional_coverage_results = conditional_coverage_test(
        kupiec_results,
        christoffersen_results
    )

    return {
        "method": method,
        "var_forecast": var_series,
        "exceptions": exceptions,
        "exception_count": exception_count,
        "exception_rate": exception_rate,
        "observations": observations,
        "kupiec": kupiec_results,
        "christoffersen": christoffersen_results,
        "conditional_coverage": conditional_coverage_results
    }

In [33]:
def compare_var_models(portfolio_returns,window_size=250,confidence_level=0.95,lambda_=0.94):
  
    methods = ["historical", "parametric", "ewma", "garch"]

    comparison = []

    for method in methods:

        results = backtest_var(
            portfolio_returns,
            method=method,
            window_size=window_size,
            confidence_level=confidence_level,
            lambda_=lambda_
        )

        comparison.append({
            "Method": method,
            "Observations": results["observations"],
            "Exceptions": results["exception_count"],
            "Exception Rate": results["exception_rate"],
            "Expected Rate": results["kupiec"]["expected_rate"],
            "Kupiec p-value": results["kupiec"]["p_value"],
            "Independence p-value": results["christoffersen"]["p_value"],
            "Conditional Coverage p-value": results["conditional_coverage"]["p_value"]
        })

    return pd.DataFrame(comparison).set_index("Method")

In [53]:
#Stress testing

In [37]:
def calculate_stress_impact(weights, shocks, portfolio_value):

    shocks = pd.Series(shocks, dtype=float)
    shocks = shocks.reindex(weights.index, fill_value=0)
    portfolio_stress_return = (weights * shocks).sum()
    stress_pnl = portfolio_value * portfolio_stress_return
    stressed_portfolio_value = portfolio_value + stress_pnl

    return {
        "stress_return": portfolio_stress_return,
        "stress_pnl": stress_pnl,
        "stressed_portfolio_value": stressed_portfolio_value
    }

In [50]:
def historical_stress_test(asset_returns,weights,portfolio_value,start_date,end_date):

    historical_returns = asset_returns.loc[start_date:end_date]

    historical_returns = historical_returns[weights.index]

    portfolio_returns = historical_returns @ weights

    portfolio_value_series = portfolio_value * (1 + portfolio_returns).cumprod()

    cumulative_return = (1 + portfolio_returns).prod() - 1

    worst_daily_return = portfolio_returns.min()

    return {
        "daily_returns": portfolio_returns,
        "portfolio_value": portfolio_value_series,
        "cumulative_return": cumulative_return,
        "worst_daily_return": worst_daily_return,
        "total_pnl": portfolio_value * cumulative_return,
        "worst_daily_pnl": portfolio_value * worst_daily_return
    }

In [41]:
def run_stress_scenarios(weights, scenarios, portfolio_value):

    results = []

    for scenario_name, shocks in scenarios.items():

        stress_result = calculate_stress_impact(
            weights,
            shocks,
            portfolio_value
        )

        results.append({
            "Scenario": scenario_name,
            "Stress Return": stress_result["stress_return"],
            "Stress P&L": stress_result["stress_pnl"],
            "Stressed Portfolio Value": stress_result["stressed_portfolio_value"]
        })

    return pd.DataFrame(results).set_index("Scenario")

In [66]:
def create_custom_scenario(weights, shocks):
    
    shocks = pd.Series(shocks, dtype=float)

    unknown_assets = shocks.index.difference(weights.index)

    if len(unknown_assets) > 0:
        raise ValueError(f"Unknown assets: {unknown_assets.tolist()}")

    if not np.isfinite(shocks).all():
        raise ValueError("Shocks must contain finite values.")

    if (shocks < -1).any():
        raise ValueError("Asset price shocks cannot be below -100%.")

    return shocks.reindex(weights.index, fill_value=0).to_dict()

In [67]:
custom_scenario = create_custom_scenario(
    weights,
    {
        "SPY": -0.25,
        "AGG": -0.08,
        "GLD": 0.15
    }
)

print(custom_scenario)

{'SPY': -0.25, 'AGG': -0.08, 'GLD': 0.15}


In [68]:

def calculate_stress_impact(weights, shocks, portfolio_value):

    if portfolio_value < 0 or not np.isfinite(portfolio_value):
        raise ValueError("portfolio_value must be finite and non-negative.")

    weights = pd.Series(weights, dtype=float)
    shocks = pd.Series(shocks, dtype=float)

    if not np.isfinite(weights).all():
        raise ValueError("Portfolio weights must be finite.")

    if not np.isfinite(shocks).all():
        raise ValueError("Shocks must be finite.")

    if (shocks < -1).any():
        raise ValueError("Asset price shocks cannot be below -100%.")

    unknown_assets = shocks.index.difference(weights.index)

    if len(unknown_assets) > 0:
        raise ValueError(f"Unknown assets: {unknown_assets.tolist()}")

    shocks = shocks.reindex(weights.index, fill_value=0)

    portfolio_stress_return = (weights * shocks).sum()
    stress_pnl = portfolio_value * portfolio_stress_return
    stressed_portfolio_value = portfolio_value + stress_pnl

    return {
        "stress_return": portfolio_stress_return,
        "stress_pnl": stress_pnl,
        "stressed_portfolio_value": stressed_portfolio_value
    }


def run_stress_scenarios(weights, scenarios, portfolio_value):

    results = []

    for scenario_name, shocks in scenarios.items():

        stress_result = calculate_stress_impact(
            weights,
            shocks,
            portfolio_value
        )

        results.append({
            "Scenario": scenario_name,
            "Stress Return": stress_result["stress_return"],
            "Stress P&L": stress_result["stress_pnl"],
            "Stressed Portfolio Value": stress_result["stressed_portfolio_value"]
        })

    return pd.DataFrame(results).set_index("Scenario")


def create_custom_scenario(weights, shocks):

    weights = pd.Series(weights, dtype=float)
    shocks = pd.Series(shocks, dtype=float)

    unknown_assets = shocks.index.difference(weights.index)

    if len(unknown_assets) > 0:
        raise ValueError(f"Unknown assets: {unknown_assets.tolist()}")

    if not np.isfinite(shocks).all():
        raise ValueError("Shocks must contain finite values.")

    if (shocks < -1).any():
        raise ValueError("Asset price shocks cannot be below -100%.")

    return shocks.reindex(weights.index, fill_value=0).to_dict()


def identify_historical_scenarios(
    asset_returns,
    weights,
    horizons=(1, 5, 20),
    top_n=5
):

    if top_n < 1:
        raise ValueError("top_n must be at least 1.")

    if not horizons or any(h < 1 for h in horizons):
        raise ValueError("All horizons must be positive integers.")

    weights = pd.Series(weights, dtype=float)

    if not np.isfinite(weights).all():
        raise ValueError("Portfolio weights must be finite.")

    if not asset_returns.columns.is_unique:
        raise ValueError("Asset return columns must be unique.")

    if not weights.index.is_unique:
        raise ValueError("Portfolio weight indices must be unique.")

    missing_assets = weights.index.difference(asset_returns.columns)

    if len(missing_assets) > 0:
        raise ValueError(f"Missing return data for assets: {missing_assets.tolist()}")

    returns = asset_returns.reindex(columns=weights.index).dropna(how="any")

    if not np.isfinite(returns.to_numpy()).all():
        raise ValueError("Asset returns must contain finite values.")

    if (returns <= -1).any().any():
        raise ValueError("Simple returns cannot be less than or equal to -100%.")

    results = {}

    for horizon in horizons:

        if len(returns) < horizon:
            raise ValueError(f"Insufficient observations for {horizon}-day horizon.")

        # Calculate compounded asset returns over rolling windows.
        asset_shocks = (
            (1 + returns)
            .rolling(window=horizon)
            .apply(np.prod, raw=True) - 1
        )

        # Calculate portfolio impact using current weights.
        portfolio_shocks = asset_shocks.mul(weights, axis=1).sum(axis=1)

        # Identify the worst portfolio scenarios.
        worst_scenarios = portfolio_shocks.nsmallest(top_n)

        scenario_results = []

        for rank, (end_date, portfolio_return) in enumerate(worst_scenarios.items(), start=1):

            end_position = returns.index.get_loc(end_date)
            start_date = returns.index[end_position - horizon + 1]

            row = {
                "Rank": rank,
                "Start Date": start_date,
                "End Date": end_date,
                "Portfolio Stress Return": portfolio_return
            }

            for asset in weights.index:
                row[f"{asset} Shock"] = asset_shocks.loc[end_date, asset]

            scenario_results.append(row)

        results[horizon] = pd.DataFrame(scenario_results).set_index("Rank")

    return results


def calculate_asset_stress_contributions(weights, shocks, portfolio_value):

    weights = pd.Series(weights, dtype=float)
    shocks = pd.Series(shocks, dtype=float)

    unknown_assets = shocks.index.difference(weights.index)

    if len(unknown_assets) > 0:
        raise ValueError(f"Unknown assets: {unknown_assets.tolist()}")

    if not np.isfinite(weights).all() or not np.isfinite(shocks).all():
        raise ValueError("Weights and shocks must contain finite values.")

    if (shocks < -1).any():
        raise ValueError("Asset price shocks cannot be below -100%.")

    if portfolio_value < 0 or not np.isfinite(portfolio_value):
        raise ValueError("portfolio_value must be finite and non-negative.")

    shocks = shocks.reindex(weights.index, fill_value=0)

    asset_pnl = portfolio_value * weights * shocks
    total_pnl = asset_pnl.sum()

    contributions = pd.DataFrame({
        "Weight": weights,
        "Stress Shock": shocks,
        "Stress P&L": asset_pnl,
        "Contribution (%)": np.where(
            total_pnl != 0,
            asset_pnl / total_pnl * 100,
            np.nan
        )
    })

    return contributions.sort_values("Stress P&L")


In [73]:
historical_scenarios = identify_historical_scenarios(
    asset_returns=daily_returns,
    weights=weights,
    horizons=(1, 5, 20),
    top_n=5
)

for horizon, scenarios in historical_scenarios.items():
    print(f"\nWorst {horizon}-day scenarios")
    display(scenarios)

ValueError: Missing return data for assets: ['AGG', 'SPY']